In [2]:
import pandas as pd
import requests
import os
from datetime import datetime
import numpy as np

# -------------------------------
# 1️⃣ CONFIG
# -------------------------------
API_KEY ="2c756e7f9e35107c881ab434f7cf06d7"
BASE_URL = "http://api.openweathermap.org/data/2.5/weather"

CITIES_FILE = "../data/raw/cities_master.csv"
RAW_FILE = "../data/raw/raw_weather_data.csv"
CLEANED_FILE = "../data/processed/clean_weather_data.csv"
FINAL_FILE = "../data/processed/climate_dashboard_dataset.csv"

# Ensure folders exist
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# -------------------------------
# 2️⃣ Fetch Data
# -------------------------------
def fetch_data():
    print("Fetching data from OpenWeather API...")

    if not os.path.exists(CITIES_FILE):
        print(f"ERROR: Cities file not found: {CITIES_FILE}")
        return

    cities_df = pd.read_csv(CITIES_FILE)

    data_list = []

    for _, row in cities_df.iterrows():
        city = row["City"]

        params = {"q": city, "appid": API_KEY, "units": "metric"}

        try:
            response = requests.get(BASE_URL, params=params, timeout=10)

            if response.status_code == 200:
                data = response.json()

                if data.get("main"):
                    data_list.append({
                        "City": city,
                        "Country": row["Country"],
                        "Region": row["Region"],
                        "Latitude": row["Latitude"],
                        "Longitude": row["Longitude"],
                        "Temperature": data["main"]["temp"],
                        "Humidity": data["main"]["humidity"],
                        "Pressure": data["main"]["pressure"],
                        "Weather_Description": data["weather"][0]["description"],
                        "Datetime": datetime.now()
                    })

        except Exception as e:
            print(f"API error for {city}: {e}")

    if not data_list:
        print("No data fetched.")
        return

    df = pd.DataFrame(data_list)

    # Overwrite RAW each run (avoid duplication)
    df.to_csv(RAW_FILE, index=False, date_format="%Y-%m-%d %H:%M:%S")

    print(f"Raw data saved: {RAW_FILE}")

# -------------------------------
# 3️⃣ Clean Data
# -------------------------------
def clean_data():
    print("Cleaning data...")

    if not os.path.exists(RAW_FILE):
        print(f"ERROR: Raw file not found: {RAW_FILE}")
        return

    df = pd.read_csv(RAW_FILE, parse_dates=["Datetime"])

    # Convert numeric columns
    for col in ["Temperature", "Humidity", "Pressure", "Latitude", "Longitude"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Handle missing values
    df["Temperature"] = df["Temperature"].fillna(df["Temperature"].mean())
    df["Humidity"] = df["Humidity"].fillna(df["Humidity"].median())
    df["Pressure"] = df["Pressure"].ffill()

    df["Latitude"] = df["Latitude"].fillna(df["Latitude"].mean())
    df["Longitude"] = df["Longitude"].fillna(df["Longitude"].mean())

    df["Weather_Description"] = df["Weather_Description"].fillna("Unknown")

    # Outlier capping
    for col in ["Temperature", "Humidity", "Pressure"]:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower, upper)

    # Remove duplicates
    df = df.drop_duplicates(subset=["City", "Datetime"], keep="last")

    df.to_csv(CLEANED_FILE, index=False, date_format="%Y-%m-%d %H:%M:%S")

    print(f"Cleaned data saved: {CLEANED_FILE}")

# -------------------------------
# 4️⃣ Aggregate for Dashboard
# -------------------------------
def aggregate_data():
    print("Aggregating data...")

    if not os.path.exists(CLEANED_FILE):
        print(f"ERROR: Cleaned file not found: {CLEANED_FILE}")
        return

    df = pd.read_csv(CLEANED_FILE, parse_dates=["Datetime"])

    df["Date"] = df["Datetime"].dt.date

    agg_df = df.groupby(
        ["City", "Country", "Region", "Latitude", "Longitude", "Date"]
    ).agg({
        "Temperature": "max",
        "Humidity": "mean",
        "Pressure": "mean"
    }).reset_index()

    agg_df.to_csv(FINAL_FILE, index=False)

    print(f"Final dataset saved: {FINAL_FILE}")

# -------------------------------
# 5️⃣ Run Pipeline
# -------------------------------
def run_pipeline():
    fetch_data()
    clean_data()
    aggregate_data()
    print("✅ Pipeline executed successfully!")

# -------------------------------
# 6️⃣ Execute
# -------------------------------
if __name__ == "__main__":
    run_pipeline()

Fetching data from OpenWeather API...
Raw data saved: ../data/raw/raw_weather_data.csv
Cleaning data...
Cleaned data saved: ../data/processed/clean_weather_data.csv
Aggregating data...
Final dataset saved: ../data/processed/climate_dashboard_dataset.csv
✅ Pipeline executed successfully!
